In [0]:
from pyspark.sql.functions import current_timestamp, col, lit

VOLUME_PATH     = "/Volumes/taxi_nyc/raw/landing/yellow/"
BRONZE_TABLE    = "taxi_nyc.bronze.yellow_trips"
CHECKPOINT_PATH = "/Volumes/taxi_nyc/bronze/pipeline_state/checkpoints/yellow/"
SCHEMA_PATH     = "/Volumes/taxi_nyc/bronze/pipeline_state/schemas/yellow/"

df_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", SCHEMA_PATH)
    .option("cloud.Files.schemaEvolutionMode", "addNewsColumns")
    .option("rescuedDataColumn", "_rescued_data")
    .load(VOLUME_PATH)
    .select(
        "*",
        col("_metadata.file_name").alias("_source_file"),
        col("_metadata.file_modification_time").alias("_modified_at"),
        current_timestamp().alias("_ingested_at"),
    )
)

(
    df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True) #availableNow=True: processa tudo que existe no momento, escreve, e encerra. O compute desliga
    .toTable(BRONZE_TABLE)
    .awaitTermination()
)

In [0]:
display(spark.table(BRONZE_TABLE).limit(10))
spark.table(BRONZE_TABLE).count()